# Session 9.4: Cloud Deployment

**Duration:** 120 minutes

## Learning Objectives
By the end of this session, you will be able to:
1. Understand different cloud platform options for ML deployment
2. Deploy a machine learning model to Heroku
3. Understand Docker basics for containerized ML applications
4. Create production-ready deployment files (Procfile, requirements.txt)
5. Handle environment variables and secrets securely
6. Test and troubleshoot deployed APIs

## Prerequisites
- Completed Sessions 9.1-9.3 (Model Serialization, Flask, FastAPI)
- Git installed and basic Git knowledge
- Heroku account (free tier)
- Working Flask or FastAPI application from previous sessions

---
## Part 1: Cloud Platforms Overview (15 minutes)

### Why Deploy to the Cloud?

Running your model locally is great for development, but to make it accessible to users, you need to deploy it to the cloud. Benefits include:

- **24/7 Availability**: Your API is always accessible
- **Scalability**: Handle multiple users simultaneously
- **Public URL**: Share your model with anyone
- **Portfolio**: Demonstrate your skills to employers

### Major Cloud Platforms

| Platform | Pros | Cons | Best For |
|----------|------|------|----------|
| **Heroku** | Easy setup, free tier, Git-based deployment | Limited free tier, cold starts | Beginners, prototypes |
| **AWS (SageMaker)** | Powerful, scalable, ML-specific tools | Complex, expensive | Enterprise ML |
| **Google Cloud (Vertex AI)** | TensorFlow integration, AutoML | Learning curve, pricing | TensorFlow projects |
| **Azure (ML Studio)** | Enterprise features, .NET integration | Complex setup | Enterprise, Microsoft stack |
| **Railway/Render** | Heroku alternatives, modern UX | Newer platforms | Modern deployments |

### This Session: Heroku

We'll use Heroku because it's:
- **Beginner-friendly**: Simple Git-based deployment
- **Free tier available**: Perfect for learning
- **Well-documented**: Extensive guides and community support
- **Production-ready**: Can scale when needed

---
## Part 2: Heroku Deployment Walkthrough (30 minutes)

### Step 1: Heroku Account and CLI Setup

**Action Items:**
1. Create account at https://signup.heroku.com/
2. Install Heroku CLI

**Installation Commands:**

In [ ]:
# macOS
!brew tap heroku/brew && brew install heroku

# Windows (use installer from https://devcenter.heroku.com/articles/heroku-cli)
# Linux
!curl https://cli-assets.heroku.com/install.sh | sh

In [ ]:
# Verify installation
!heroku --version

In [ ]:
# Login to Heroku (opens browser)
!heroku login

### Step 2: Understanding Deployment Files

#### requirements.txt
Lists all Python dependencies with specific versions for reproducibility.

In [ ]:
%%writefile stock_predictor_deployment/requirements.txt
flask==3.0.0
gunicorn==21.2.0
scikit-learn==1.3.2
pandas==2.1.4
numpy==1.26.2
joblib==1.3.2

#### Procfile
Tells Heroku how to run your application. Uses Gunicorn (production WSGI server).

In [ ]:
%%writefile stock_predictor_deployment/Procfile
web: gunicorn app:app

#### runtime.txt (Optional)
Specifies Python version.

In [ ]:
%%writefile stock_predictor_deployment/runtime.txt
python-3.11.7

#### .gitignore
Files to exclude from Git (virtual environments, cached files, secrets).

In [ ]:
%%writefile stock_predictor_deployment/.gitignore
venv/
__pycache__/
*.pyc
.env
.DS_Store
*.pkl
*.joblib

### Step 3: Deployment Process

The deployment workflow looks like this:

```
Local Development → Git Commit → Push to Heroku → Automatic Build → Live App
```

In [ ]:
# Navigate to your app directory
import os
os.chdir('stock_predictor_deployment')

# Initialize Git repository
!git init
!git add .
!git commit -m "Initial commit: Stock predictor API"

In [ ]:
# Create Heroku app (generates random name or use your own)
!heroku create stock-predictor-ml-app

# This creates a new Heroku app and adds a Git remote named 'heroku'

In [ ]:
# Deploy to Heroku
!git push heroku main

# This triggers:
# 1. Code upload
# 2. Python environment setup
# 3. Dependency installation
# 4. Application start

In [ ]:
# Check if app is running
!heroku ps

# View logs
!heroku logs --tail

In [ ]:
# Open your app in browser
!heroku open

---
## Part 3: Docker Basics for ML (25 minutes)

### What is Docker?

Docker packages your application and all its dependencies into a **container** - a standardized unit that runs consistently anywhere.

**Containers vs Virtual Machines:**

| Feature | Containers | Virtual Machines |
|---------|-----------|------------------|
| Size | Lightweight (MB) | Heavy (GB) |
| Startup | Seconds | Minutes |
| Performance | Near-native | Slower |
| Isolation | Process-level | Full OS |

### Why Docker for ML?

1. **Reproducibility**: "Works on my machine" → "Works everywhere"
2. **Dependency Management**: No conflicts between projects
3. **Deployment**: Same container from dev to production
4. **Scalability**: Easy to replicate and scale

### Dockerfile Anatomy

In [ ]:
%%writefile stock_predictor_deployment/Dockerfile
# Base image with Python
FROM python:3.11-slim

# Set working directory
WORKDIR /app

# Copy requirements first (for caching)
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose port
EXPOSE 5000

# Run application
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "app:app"]

### Building and Running Docker Containers

In [ ]:
# Build Docker image
!docker build -t stock-predictor:v1 .

# Run container locally
!docker run -p 5000:5000 stock-predictor:v1

# List running containers
!docker ps

### Docker + Heroku (Advanced)

Heroku also supports Docker deployments via Container Registry:

In [ ]:
# Login to Heroku Container Registry
!heroku container:login

# Build and push Docker image
!heroku container:push web -a your-app-name

# Release the image
!heroku container:release web -a your-app-name

### LLM Prompt for Docker Configuration

Use this prompt with ChatGPT/Claude to generate Dockerfiles:

```
Create a production-ready Dockerfile for a Flask ML API with these requirements:
- Python 3.11
- Dependencies: flask, scikit-learn, pandas, numpy, gunicorn
- Model file: stock_model.pkl (30MB)
- Optimize for small image size
- Use multi-stage build if beneficial
- Include health check
```

---
## Part 4: Stock Predictor Deployment Hands-On (40 minutes)

Now let's deploy our complete stock price predictor to Heroku!

### Step 1: Prepare the Flask Application

We'll create a complete Flask app that loads our trained model and serves predictions.

In [ ]:
# First, let's train a simple stock predictor model
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import joblib
from datetime import datetime, timedelta

# Generate synthetic stock data for demonstration
np.random.seed(42)
dates = pd.date_range(end=datetime.now(), periods=500, freq='D')

# Create features: day of year, day of week, trend
data = pd.DataFrame({
    'date': dates,
    'day_of_year': dates.dayofyear,
    'day_of_week': dates.dayofweek,
    'trend': range(500)
})

# Generate price with trend and seasonality
base_price = 100
trend = data['trend'] * 0.1
seasonality = 10 * np.sin(2 * np.pi * data['day_of_year'] / 365)
noise = np.random.normal(0, 5, 500)
data['price'] = base_price + trend + seasonality + noise

# Create lagged features (previous prices)
data['price_lag_1'] = data['price'].shift(1)
data['price_lag_7'] = data['price'].shift(7)
data['price_lag_30'] = data['price'].shift(30)
data['rolling_mean_7'] = data['price'].rolling(window=7).mean()
data['rolling_std_7'] = data['price'].rolling(window=7).std()

# Drop rows with NaN
data = data.dropna()

# Prepare training data
feature_cols = ['day_of_year', 'day_of_week', 'trend', 'price_lag_1', 
                'price_lag_7', 'price_lag_30', 'rolling_mean_7', 'rolling_std_7']
X = data[feature_cols]
y = data['price']

# Split data
split_idx = int(len(data) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"Train R²: {train_score:.3f}")
print(f"Test R²: {test_score:.3f}")

# Save model
joblib.dump(model, 'stock_predictor_deployment/stock_model.pkl')
print("\nModel saved successfully!")

### Step 2: Create the Flask API

The app.py file is created in the `stock_predictor_deployment/` folder. Let's review its structure:

In [ ]:
# View the app.py file
!cat stock_predictor_deployment/app.py

### Step 3: Test Locally Before Deployment

In [ ]:
# Test locally with gunicorn (same as Heroku will use)
!cd stock_predictor_deployment && gunicorn --bind 0.0.0.0:5000 app:app

In [ ]:
# In a separate terminal or cell, test the API
import requests

# Test health check
response = requests.get('http://localhost:5000/health')
print("Health Check:", response.json())

# Test prediction
test_data = {
    "day_of_year": 350,
    "day_of_week": 3,
    "trend": 500,
    "price_lag_1": 195.5,
    "price_lag_7": 193.2,
    "price_lag_30": 188.7,
    "rolling_mean_7": 194.3,
    "rolling_std_7": 2.1
}

response = requests.post('http://localhost:5000/predict', json=test_data)
print("\nPrediction:", response.json())

### Step 4: Deploy to Heroku

Follow these commands in order:

In [ ]:
%%bash
cd stock_predictor_deployment

# Initialize Git
git init
git add .
git commit -m "Deploy stock predictor API"

# Create Heroku app
heroku create ml-stock-predictor-$(date +%s)

# Deploy
git push heroku main

# Check status
heroku ps

# View logs
heroku logs --tail

### Step 5: Test the Deployed API

In [ ]:
# Get your Heroku app URL
import subprocess
result = subprocess.run(['heroku', 'apps:info', '-s'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if line.startswith('web_url='):
        app_url = line.split('=')[1]
        print(f"App URL: {app_url}")
        break

# Test the deployed API
import requests

base_url = app_url.strip('/')

# Health check
response = requests.get(f"{base_url}/health")
print("\nHealth Check:", response.json())

# Make prediction
test_data = {
    "day_of_year": 350,
    "day_of_week": 3,
    "trend": 500,
    "price_lag_1": 195.5,
    "price_lag_7": 193.2,
    "price_lag_30": 188.7,
    "rolling_mean_7": 194.3,
    "rolling_std_7": 2.1
}

response = requests.post(f"{base_url}/predict", json=test_data)
print("\nPrediction:", response.json())

print(f"\n🎉 Success! Your API is live at: {base_url}")

---
## Part 5: Production Considerations (15 minutes)

### Environment Variables & Secrets

**Never hardcode secrets in code!** Use environment variables instead.

In [ ]:
# Set environment variables on Heroku
!heroku config:set API_KEY=your-secret-key
!heroku config:set MODEL_VERSION=v1.0.0
!heroku config:set DEBUG=False

# View config
!heroku config

In [ ]:
# Use in your Flask app
import os

# Good practice: use environment variables
API_KEY = os.environ.get('API_KEY', 'default-dev-key')
MODEL_VERSION = os.environ.get('MODEL_VERSION', 'v1.0.0')
DEBUG = os.environ.get('DEBUG', 'True') == 'True'

# Example: protect endpoint with API key
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    # Check API key
    provided_key = request.headers.get('X-API-Key')
    if provided_key != API_KEY:
        return jsonify({'error': 'Invalid API key'}), 401
    
    # Process prediction...
    return jsonify({'prediction': 123.45})

### Scaling Considerations

#### Vertical Scaling (Bigger Dynos)
Increase resources of a single instance.

In [ ]:
# Upgrade dyno type (requires paid plan)
!heroku ps:scale web=1:standard-1x

#### Horizontal Scaling (More Dynos)
Run multiple instances in parallel.

In [ ]:
# Scale to 3 dynos
!heroku ps:scale web=3

### Cold Starts

**Problem:** Free Heroku dynos sleep after 30 minutes of inactivity. First request takes 10-30 seconds.

**Solutions:**
1. **Upgrade to Hobby tier** ($7/month) - no sleeping
2. **Keep-alive service** - Ping your app every 25 minutes
3. **Optimize startup time** - Lazy load models only when needed

In [ ]:
# Lazy loading example
model = None

def get_model():
    global model
    if model is None:
        print("Loading model...")
        model = joblib.load('stock_model.pkl')
    return model

@app.route('/predict', methods=['POST'])
def predict():
    model = get_model()  # Only loads on first request
    # Make prediction...
    return jsonify({'prediction': 123.45})

### Cost Considerations

| Tier | Cost | Features | Best For |
|------|------|----------|----------|
| **Free** | $0/month | 550-1000 dyno hours/month, sleeps after 30min | Learning, demos |
| **Hobby** | $7/month | Always on, SSL, custom domains | Side projects |
| **Standard** | $25-50/month | Better performance, horizontal scaling | Production apps |
| **Performance** | $250-500/month | High memory, dedicated resources | Enterprise ML |

**Free Tier Limits:**
- Max 512MB RAM
- Sleeps after 30min inactivity
- Limited to 1000 dyno hours/month (verified account)

**Tip:** Use free tier for development, upgrade when you have users.

### Security Best Practices

1. **Use HTTPS only** (Heroku provides free SSL)
2. **Validate inputs** (prevent injection attacks)
3. **Rate limiting** (prevent abuse)
4. **API authentication** (API keys or tokens)
5. **Don't log sensitive data**
6. **Keep dependencies updated**

In [ ]:
# Example: Input validation
from flask import request, jsonify

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    
    # Validate required fields
    required_fields = ['day_of_year', 'day_of_week', 'trend', 
                      'price_lag_1', 'price_lag_7', 'price_lag_30',
                      'rolling_mean_7', 'rolling_std_7']
    
    for field in required_fields:
        if field not in data:
            return jsonify({'error': f'Missing field: {field}'}), 400
    
    # Validate types and ranges
    try:
        day_of_year = int(data['day_of_year'])
        if not 1 <= day_of_year <= 366:
            return jsonify({'error': 'day_of_year must be 1-366'}), 400
    except (ValueError, TypeError):
        return jsonify({'error': 'Invalid day_of_year'}), 400
    
    # Proceed with prediction...
    return jsonify({'prediction': 123.45})

---
## Troubleshooting Common Issues

### Issue 1: Application Error (H10)

**Symptom:** "Application Error" when visiting URL

**Solution:**

In [ ]:
# Check logs
!heroku logs --tail

# Common causes:
# 1. Procfile missing or incorrect
# 2. Port binding issue (use PORT env variable)
# 3. Missing dependencies in requirements.txt
# 4. Python version mismatch

### Issue 2: Module Not Found

**Symptom:** `ModuleNotFoundError: No module named 'sklearn'`

**Solution:**

In [ ]:
# Generate requirements.txt from current environment
!pip freeze > requirements.txt

# Or manually add the missing package
# Edit requirements.txt and add:
# scikit-learn==1.3.2

# Commit and redeploy
!git add requirements.txt
!git commit -m "Add missing dependency"
!git push heroku main

### Issue 3: Slug Size Too Large

**Symptom:** "Compiled slug size: 550MB is too large (max is 500MB)"

**Solution:**

In [ ]:
# 1. Use .slugignore to exclude files
%%writefile .slugignore
*.ipynb
tests/
notebooks/
*.csv
*.h5
data/

# 2. Use lighter dependencies
# Instead of: tensorflow==2.14.0 (500MB+)
# Use: tensorflow-cpu==2.14.0 (smaller)

# 3. Store large models externally (S3, Google Drive)
# Download at startup instead of including in slug

### Issue 4: Memory Exceeded (R14/R15)

**Symptom:** App crashes with "Memory quota exceeded"

**Solution:**

In [ ]:
# 1. Monitor memory usage
!heroku logs --tail | grep "R14\|R15"

# 2. Optimize model size
# - Use model compression
# - Load model only when needed
# - Use lighter model architecture

# 3. Upgrade dyno (Hobby has 512MB, Standard-1X has 512MB, Standard-2X has 1GB)
!heroku ps:scale web=1:standard-2x

---
## LLM Prompts for Deployment

### Prompt 1: Generate Deployment Files

In [ ]:
prompt_1 = """
Generate a complete Heroku deployment setup for a Flask ML API with these specifications:

Application Details:
- Framework: Flask 3.0
- ML Libraries: scikit-learn, pandas, numpy
- Model: Random Forest saved with joblib
- Python version: 3.11

Please create:
1. Procfile for Gunicorn
2. requirements.txt with pinned versions
3. runtime.txt
4. .gitignore
5. Basic app.py with health check and predict endpoints

Include error handling and logging.
"""

print(prompt_1)

### Prompt 2: Optimize Dockerfile

In [ ]:
prompt_2 = """
Optimize this Dockerfile for a production ML API:

Current Dockerfile:
FROM python:3.11
COPY . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD ["python", "app.py"]

Requirements:
- Minimize image size
- Improve build caching
- Use multi-stage build if beneficial
- Add health check
- Use non-root user for security
- Optimize for scikit-learn (avoid unnecessary dependencies)

Provide the optimized Dockerfile with explanations.
"""

print(prompt_2)

### Prompt 3: Troubleshoot Deployment Error

In [ ]:
prompt_3 = """
My Heroku deployment is failing with this error:

```
2024-01-06T10:30:45.123456+00:00 app[web.1]: ModuleNotFoundError: No module named 'sklearn'
2024-01-06T10:30:45.234567+00:00 heroku[web.1]: State changed from starting to crashed
```

My requirements.txt contains:
```
flask==3.0.0
gunicorn==21.2.0
scikit-learn==1.3.2
```

Please help me:
1. Diagnose the issue
2. Provide step-by-step fix
3. Explain why this happened
4. Suggest prevention strategies
"""

print(prompt_3)

---
## Summary & Next Steps

### What You Learned

✅ Cloud platform options and when to use each
✅ Heroku deployment workflow (Git → Deploy → Monitor)
✅ Creating deployment files (Procfile, requirements.txt, runtime.txt)
✅ Docker basics for containerized ML applications
✅ Production considerations (secrets, scaling, costs)
✅ Troubleshooting common deployment issues

### Key Takeaways

1. **Start Simple:** Heroku for beginners, AWS/GCP for scale
2. **Test Locally First:** Use gunicorn locally before deploying
3. **Environment Variables:** Never hardcode secrets
4. **Monitor Logs:** `heroku logs --tail` is your best friend
5. **Start Free, Scale Up:** Use free tier to learn, upgrade when needed

### Portfolio Checklist

- [ ] Working public API URL
- [ ] Clear API documentation
- [ ] Example requests/responses
- [ ] GitHub repository with deployment guide
- [ ] Error handling and validation
- [ ] Environment variables for configuration

### Next Session Preview

**Session 9.5: Model Monitoring**
- Track prediction requests and responses
- Detect model degradation (data drift, concept drift)
- Build monitoring dashboards with Streamlit
- Set up alerts for anomalies

### Resources

- [Heroku Python Documentation](https://devcenter.heroku.com/categories/python-support)
- [Docker for ML Tutorial](https://mlinproduction.com/docker-for-ml-part-1/)
- [Deploying ML Models Guide](https://christophergs.com/machine%20learning/2019/03/17/how-to-deploy-machine-learning-models/)
- [Flask Production Best Practices](https://flask.palletsprojects.com/en/3.0.x/deploying/)

---
## Practice Exercises

### Exercise 1: Deploy Your Own Model (30 minutes)

Deploy one of your previous models (spam detector, image classifier, etc.) to Heroku.

**Requirements:**
1. Create Flask API with /health and /predict endpoints
2. Add input validation
3. Include error handling
4. Deploy to Heroku
5. Test with real requests
6. Write API documentation

### Exercise 2: Dockerize Your Application (20 minutes)

Create a Dockerfile for your API and run it locally.

**Requirements:**
1. Write Dockerfile
2. Build image
3. Run container
4. Test endpoints
5. Push to Docker Hub (optional)

### Exercise 3: Add Security Features (15 minutes)

Enhance your deployed API with security features.

**Requirements:**
1. Add API key authentication
2. Implement rate limiting
3. Add request validation
4. Use environment variables for secrets
5. Test security features